# VoxaVersa: train all AI voices on a free Colab GPU

This notebook runs **the same training queue** as VoxaVersa's CPU trainer, on Google's GPU: same data, same
quality checks every 10 epochs, same automatic stop and best-checkpoint choice, and the same matching of every
singer to the built-in genre voices. It is just much faster (about 1 - 3 hours per voice instead of 14 - 45).

**Before you start** (once): put `VoxaVersa_Colab_Data.zip` in your Google Drive, in a folder named
`VoxaVersa` (so: *My Drive > VoxaVersa > VoxaVersa_Colab_Data.zip*). Don't unzip it. It is private
test/training material: don't share it.

**Every time**

1. Menu **Runtime > Change runtime type > T4 GPU > Save**.
2. Menu **Runtime > Run all**. Allow access to Google Drive when asked.
3. Step 6 shows the progress. Leave the tab open. If Colab disconnects or the daily GPU time runs out, come
   back later and do **Run all** again: it continues where it stopped (from the last checkpoint on Drive).

Finished voices appear in *My Drive > VoxaVersa > queue > voices* as `.vxpack` files (step 7 lists them and
plays before/after samples). Step-by-step guide: `docs/COLAB.md` in the VoxaVersa repository.

In [ ]:
#@title 1. Check the GPU
import subprocess
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
if gpu:
    print("OK - GPU found:", gpu)
else:
    raise SystemExit("No GPU. Menu Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all again.")

In [ ]:
#@title 2. Settings and Google Drive
#@markdown **Training data kit** in your Google Drive (path inside *My Drive*):
DATA_KIT = "VoxaVersa/VoxaVersa_Colab_Data.zip"  #@param {type:"string"}
#@markdown Also train the 20 free VocalSet singers (downloaded here; CC BY 4.0). Untick to train only your studio singers.
INCLUDE_VOCALSET = True  #@param {type:"boolean"}
#@markdown Precision: *mixed* is RVC's standard GPU training (faster); *full* is fp32 like the CPU trainer (~2x slower).
PRECISION = "mixed"  #@param ["mixed", "full"]
BATCH_SIZE = 8  #@param {type:"integer"}
#@markdown Where the VoxaVersa tools come from (leave as is unless told otherwise):
VOXAVERSA_REPO = "https://github.com/DK7602/VoxaVersa"  #@param {type:"string"}
VOXAVERSA_BRANCH = "claude/voxaversa-voice-plugin-20arm6"  #@param {type:"string"}

import os
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/VoxaVersa/queue"
KIT = os.path.join("/content/drive/MyDrive", DATA_KIT)
os.makedirs(WORK, exist_ok=True)
if not os.path.exists(KIT):
    raise SystemExit("Not found in your Drive: My Drive/%s\nUpload VoxaVersa_Colab_Data.zip there first." % DATA_KIT)
print("OK - data kit found; results and checkpoints go to My Drive/VoxaVersa/queue")

In [ ]:
#@title 3. Install the training software (about 5 minutes the first time)
#@markdown Also fetches the newest VoxaVersa tools, so fixes arrive without a new notebook.
import os, subprocess, sys, importlib
REPO = "/content/VoxaVersa"
if not os.path.isfile(REPO + "/tools/training/colab_steps.py"):
    subprocess.run("rm -rf %s && git clone -q --depth 1 --branch %s %s %s" % (REPO, VOXAVERSA_BRANCH, VOXAVERSA_REPO, REPO),
                   shell=True, check=True)
else:
    subprocess.run("git -C %s pull -q" % REPO, shell=True, check=True)
TOOLS = REPO + "/tools/training"
sys.path.insert(0, TOOLS)
import colab_steps
importlib.reload(colab_steps)
colab_steps.install()

In [ ]:
#@title 4. Trainer, base models and content encoder (2 - 5 minutes the first time)
colab_steps.setup(WORK)

In [ ]:
#@title 5. Training data (seconds; the 20 VocalSet singers download in the background, see step 6)
colab_steps.data(WORK, KIT, vocalset=INCLUDE_VOCALSET, precision=PRECISION, batch=BATCH_SIZE)

In [ ]:
#@title 6. Train (runs until every voice is finished; keep this tab open)
#@markdown Shows the progress every 2 minutes. You can stop this cell any time; training keeps going in the
#@markdown background while Colab stays connected. Run all again after a disconnect to continue.
colab_steps.train(WORK)

In [ ]:
#@title 7. Finished voices: listen and download
colab_steps.results(WORK)

## Using the voices in VoxaVersa

Download the folder *My Drive > VoxaVersa > queue > voices* (right-click > Download), then follow
**docs/COLAB.md, part 3**: install the content encoder `contentvec-768.onnx` (in *My Drive > VoxaVersa >
queue*) once, and import each `.vxpack` in VoxaVersa's Voice Library. Each pack already says which built-in
genre voices it powers.